In [1]:
import os
import json
import time
import pandas as pd
from tqdm.auto import tqdm

from raptor import RetrievalAugmentation
from raptor import RetrievalAugmentationConfig

from custom_models.embedding import MiniLMEmbedding
from custom_models.summarizer import MistralSummarizer
from custom_models.qa import MistralQA


from custom_models.embedding import MiniLMEmbedding
from custom_models.summarizer import MistralSummarizer
from custom_models.qa import MistralQA

from raptor import RetrievalAugmentation, RetrievalAugmentationConfig

c:\Users\apurb_oi8roye\anaconda3\envs\raptor2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-06 20:29:39,513 - Loading faiss with AVX2 support.
2026-07-06 20:29:39,517 - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-07-06 20:29:39,517 - Loading faiss.
2026-07-06 20:29:39,671 - Successfully loaded faiss.


In [2]:
QASPER_DIR = r"C:\Users\apurb_oi8roye\Desktop\RAG_Research_Summary -new\models\qasper_data"

DEV_PATH = os.path.join(QASPER_DIR, "qasper-dev-v0.3.json")

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

print("Validation papers:", len(dev_data))

Validation papers: 281


In [3]:
embedding = MiniLMEmbedding()
summarizer = MistralSummarizer()
qa = MistralQA()

config = RetrievalAugmentationConfig(
    embedding_model=embedding,
    summarization_model=summarizer,
    qa_model=qa,
    tb_max_tokens=100,     # RAPTOR paper
    tb_num_layers=5,

    tr_top_k=20 
)

2026-07-03 16:38:29,260 - No device provided, using cpu
2026-07-03 16:38:29,931 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-03 16:38:29,999 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-03 16:38:30,305 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-03 16:38:30,311 - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-03 16:38:30,378 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 

In [4]:
def paper_to_text(paper):

    text = paper["title"] + "\n\n"

    text += paper["abstract"] + "\n\n"

    for sec in paper["full_text"]:

        text += sec["section_name"] + "\n"

        for para in sec["paragraphs"]:
            text += para + "\n"

        text += "\n"

    return text

In [5]:
OUTPUT = "raptor_validation_predictions.csv"

if os.path.exists(OUTPUT):

    results = pd.read_csv(OUTPUT)

    completed = set(results.paper_id.unique())

    print("Already processed:", len(completed))

else:

    results = pd.DataFrame()

    completed = set()

Already processed: 1


In [6]:
import os
import time
import pandas as pd
from tqdm.auto import tqdm

OUTPUT = "raptor_validation_predictions.csv"

# =====================================================
# Resume if output already exists
# =====================================================

if os.path.exists(OUTPUT):
    old = pd.read_csv(OUTPUT)
    completed = set(old.paper_id.unique())
    print(f"Resuming from {len(completed)} completed papers.")
else:
    completed = set()
    print("Starting a fresh evaluation.")

results = []

# =====================================================
# Choose papers
# =====================================================

# Full validation set
papers = list(dev_data.items())

# Uncomment instead if you only want first 50 papers
# papers = list(dev_data.items())[:50]

# =====================================================
# Evaluation
# =====================================================

for paper_id, paper in tqdm(papers):

    if paper_id in completed:
        continue

    paper_text = paper_to_text(paper)

    # ---------- Build RAPTOR tree ----------
    build_start = time.time()

    RA = RetrievalAugmentation(config=config)

    RA.add_documents(paper_text)

    build_time = time.time() - build_start

    leaf_nodes = len(RA.tree.leaf_nodes)
    root_nodes = len(RA.tree.root_nodes)
    total_nodes = len(RA.tree.all_nodes)

    # ---------- Answer every question ----------
    for qa_item in paper["qas"]:

        question = qa_item["question"]

        qa_start = time.time()

        prediction = RA.answer_question(
            question,
            max_tokens=2000
        )

        qa_time = time.time() - qa_start

        gold = qa_item["answers"][0]["answer"]

        results.append({

            "paper_id": paper_id,

            "question_id": qa_item["question_id"],

            "question": question,

            "prediction": prediction,

            "gold_extract": gold["extractive_spans"],

            "gold_freeform": gold["free_form_answer"],

            "build_time": build_time,

            "qa_time": qa_time,

            "leaf_nodes": leaf_nodes,

            "root_nodes": root_nodes,

            "total_nodes": total_nodes

        })

    # ---------- Save after every paper ----------
    df = pd.DataFrame(results)

    if os.path.exists(OUTPUT):
        df.to_csv(
            OUTPUT,
            mode="a",
            header=False,
            index=False
        )
    else:
        df.to_csv(
            OUTPUT,
            index=False
        )

    # Clear memory
    results = []

print("Finished!")

  0%|          | 0/10 [00:00<?, ?it/s]2026-07-03 16:38:38,709 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 100
            Summarization Model: <custom_models.summarizer.MistralSummarizer object at 0x00000275D7F64250>
            Embedding Models: {'EMB': <custom_models.embedding.MiniLMEmbedding object at 0x00000275D7F648B0>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-07-03 16:38:38,711 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
        

In [7]:
import pandas as pd

df = pd.read_csv("raptor_validation_predictions.csv")

print(df.shape)
df.head()

(34, 11)


,paper_id,question_id,question,prediction,gold_extract,gold_freeform,build_time,qa_time,leaf_nodes,root_nodes,total_nodes
0,1912.01214,b6f15fb6279b82e34a5bf4828b7b5ddabfdf1d54,which multilingual approaches do they compare ...,The text compares several multilingual approa...,"['BIBREF19', 'BIBREF20']",NaN,332.863156,5.169714,70,2,85
1,1912.01214,f5e6f43454332e0521a778db0b769481e23e7682,what are the pivot-based baselines?,The pivot-based baselines refer to methods wh...,"['pivoting', 'pivoting$_{\\rm m}$']",NaN,332.863156,3.617219,70,2,85
2,1912.01214,9a05a5f4351db75da371f7ac12eb0b03607c4b87,which datasets did they experiment with?,The experiments were conducted on Europarl an...,"['Europarl', 'MultiUN']",NaN,332.863156,3.067660,70,2,85
3,1912.01214,5eda469a8a77f028d0c5f1acd296111085614537,what language pairs are explored?,The language pairs explored include French-En...,[],"De-En, En-Fr, Fr-En, En-Es, Ro-En, En-De, Ar-E...",332.863156,6.772643,70,2,85
4,1912.01214,b6f15fb6279b82e34a5bf4828b7b5ddabfdf1d54,which multilingual approaches do they compare ...,They compare their approach with traditional ...,"['BIBREF19', 'BIBREF20']",NaN,326.524418,3.588987,70,3,85


In [8]:
print("Questions:", len(df))

print("Average build time:",
      df.build_time.mean())

print("Average QA time:",
      df.qa_time.mean())

print("Average leaf nodes:",
      df.leaf_nodes.mean())

print("Average root nodes:",
      df.root_nodes.mean())

Questions: 34
Average build time: 255.453437945422
Average QA time: 3.95443435977487
Average leaf nodes: 66.70588235294117
Average root nodes: 5.0


In [9]:
import re
import string
from collections import Counter

In [16]:
import pandas as pd
import json

df = pd.read_csv("raptor_validation_predictions.csv")

with open("raptor_predictions.jsonl", "w", encoding="utf-8") as f:

    for _, row in df.iterrows():

        pred = {
            "question_id": row["question_id"],
            "predicted_answer": "" if pd.isna(row["prediction"]) else str(row["prediction"]),
            "predicted_evidence": []
        }

        f.write(json.dumps(pred) + "\n")

print("Saved", len(df), "predictions.")

Saved 34 predictions.


In [21]:
import subprocess

result = subprocess.run(
    [
        "python",
        r"C:\Users\apurb_oi8roye\Desktop\qasper-led-baseline\scripts\evaluator.py",
        "--predictions",
        "raptor_predictions.jsonl",
        "--gold",
        r"C:\Users\apurb_oi8roye\Desktop\RAG_Research_Summary -new\models\qasper_data\qasper-dev-v0.3.json",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

{
  "Answer F1": 0.30466871648130706,
  "Answer F1 by type": {
    "extractive": 0.3439109244745849,
    "abstractive": 0.19402343158884014,
    "boolean": 0.05505279034690799,
    "none": 0.0
  },
  "Evidence F1": 0.06666666666666667,
  "Missing predictions": 0
}

